In [147]:
import pandas as pd
import numpy as np

# Labels/ ground truth

In [148]:
hormones_and_selfreport=pd.read_csv("hormones_and_selfreport.csv")

In [149]:
hormones_and_selfreport.head()

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,headaches,cramps,sorebreasts,fatigue,sleepissue,moodswing,stress,foodcravings,indigestion,bloating
0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,Not at all,...,High,Very Low/Little,Very Low/Little,High,Low,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,Not at all,...,Very High,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,Not at all,...,High,Very Low/Little,Very Low/Little,Very High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,Not at all,...,Very Low/Little,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,Not at all,...,Very Low/Little,Very Low/Little,Very Low/Little,High,High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little


In [150]:
hormones_and_selfreport.columns

Index(['id', 'study_interval', 'is_weekend', 'day_in_study', 'phase', 'lh',
       'estrogen', 'pdg', 'flow_volume', 'flow_color', 'appetite',
       'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 'fatigue',
       'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion',
       'bloating'],
      dtype='str')

In [151]:
hormones_and_selfreport_short=hormones_and_selfreport[['id', 'study_interval','day_in_study',
                                                        'phase', 'lh','estrogen', 'pdg', 'flow_volume']]

In [ ]:
(set(hormones_and_selfreport_short['id'].to_list()))

In [7]:
hormones_and_selfreport_short.head(1)

,id,study_interval,day_in_study,phase,lh,estrogen,pdg,flow_volume
0,1,2022,1,Follicular,2.9,94.2,NaN,Not at all


In [70]:
print(hormones_and_selfreport_short["flow_volume"].unique())   
FLOW = {
    "Not at all": 0,
    "Spotting / Very Light": 1,
    "Somewhat Light": 2,
    "Light": 2,
    "Moderate": 3,
    "Somewhat Heavy": 4,
    "Heavy": 5,
}
hormones_and_selfreport_short["flow"] = hormones_and_selfreport_short["flow_volume"].map(FLOW)
print("still unmapped:", hormones_and_selfreport_short.loc[hormones_and_selfreport_short.flow.isna() & hormones_and_selfreport_short.flow_volume.notna(), "flow_volume"].unique())

<ArrowStringArray>
[           'Not at all',                     nan,            'Very Heavy',
        'Somewhat Heavy',              'Moderate',                 'Light',
 'Spotting / Very Light',                 'Heavy',        'Somewhat Light']
Length: 9, dtype: str
still unmapped: <ArrowStringArray>
['Very Heavy']
Length: 1, dtype: str


In [71]:
FLOW = {
    "Not at all": 0,
    "Spotting / Very Light": 1,
    "Somewhat Light": 2,
    "Light": 2,
    "Moderate": 3,
    "Somewhat Heavy": 4,
    "Heavy": 5,
    "Very Heavy": 6,
}
hormones_and_selfreport_short["flow"] = hormones_and_selfreport_short["flow_volume"].map(FLOW)
print("still unmapped:", hormones_and_selfreport_short.loc[hormones_and_selfreport_short.flow.isna() & hormones_and_selfreport_short.flow_volume.notna(), "flow_volume"].unique())

still unmapped: <ArrowStringArray>
[]
Length: 0, dtype: str


In [76]:
# finds first day of each period
import pandas as pd
KEY = ["id", "study_interval", "day_in_study"]

def detect_menses_onsets(hormones_and_selfreport_short, min_gap=10):
    df = hormones_and_selfreport_short.sort_values(KEY).copy()
    df["bleeding"] = df["flow"].fillna(0) > 0
    onsets = []
    for (pid, interval), g in df.groupby(["id", "study_interval"], sort=False):
        g = g.sort_values("day_in_study")
        last_onset = -999                         
        prev = False
        for d, b in zip(g["day_in_study"], g["bleeding"]):
            if b and not prev and (d - last_onset) >= min_gap:
                onsets.append({"id": pid, "study_interval": interval, "onset_day": int(d)})
                last_onset = d                 
            prev = b
    return pd.DataFrame(onsets)

In [77]:
# finds ovulation by locating surge in LH hormone
def detect_ovulation_from_hormones(hormones_and_selfreport_short):
    onsets = detect_menses_onsets(hormones_and_selfreport_short)
    rows = []
    for (pid, interval), g_on in onsets.groupby(["id", "study_interval"], sort=False):
        od = sorted(g_on["onset_day"].tolist())
        hg = hormones_and_selfreport_short[
            (hormones_and_selfreport_short.id == pid) &
            (hormones_and_selfreport_short.study_interval == interval)]
        lh_base = hg["lh"].median()                      # this person's LH baseline
        for a, b in zip(od, od[1:]):                     # each closed cycle [a, b)
            cyc_len = b - a
            lo, hi = a + int(cyc_len * 0.25), a + int(cyc_len * 0.75)   # mid-cycle only
            win = hg[(hg.day_in_study >= lo) & (hg.day_in_study <= hi)]
            lh = win["lh"].dropna()
            if lh.empty:
                continue
            if lh.max() < lh_base * 1.5:                 # require a real surge, else skip
                continue
            ov_day = int(win.loc[win["lh"].idxmax(), "day_in_study"])
            post = win[win.day_in_study > ov_day]["pdg"]
            pre  = win[win.day_in_study <= ov_day]["pdg"]
            confirmed = (post.mean() > pre.mean()) if post.notna().any() and pre.notna().any() else False
            rows.append({"id": pid, "study_interval": interval,
                         "onset_day": a, "next_onset_day": b,
                         "ovulation_day": ov_day, "pdg_confirmed": confirmed})
    return pd.DataFrame(rows)

In [ ]:
# length of each stage
def build_cycle_sequence(events):
    df = events.copy()
    df["cycle_length"]      = df["next_onset_day"] - df["onset_day"]
    df["follicular_length"] = df["ovulation_day"]  - df["onset_day"]  
    df["luteal_length"]     = df["cycle_length"]   - df["follicular_length"]
    # drop impossible parses
    df = df[df.cycle_length.between(18, 60) &
            df.follicular_length.between(5, 35) &
            df.luteal_length.between(7, 20)].copy()
    df = df.sort_values(["id", "study_interval", "onset_day"]).reset_index(drop=True)
    df["cycle_index"] = df.groupby("id").cumcount()
    # causal personal history (row j sees only cycles 0..j-1)
    grp = df.groupby("id", sort=False)
    for col in ["cycle_length", "follicular_length", "luteal_length"]:
        df[f"prior_{col}_mean"] = grp[col].transform(lambda s: s.expanding().mean().shift(1))
        df[f"prior_{col}_sd"]   = grp[col].transform(lambda s: s.expanding().std().shift(1))
    df["n_prior_cycles"] = df["cycle_index"]
    return df

In [78]:
onsets    = detect_menses_onsets(hormones_and_selfreport_short)
events    = detect_ovulation_from_hormones(hormones_and_selfreport_short)
cycle_seq = build_cycle_sequence(events)
cycle_seq.head()

,id,study_interval,onset_day,next_onset_day,ovulation_day,pdg_confirmed,cycle_length,follicular_length,luteal_length,cycle_index,prior_cycle_length_mean,prior_cycle_length_sd,prior_follicular_length_mean,prior_follicular_length_sd,prior_luteal_length_mean,prior_luteal_length_sd,n_prior_cycles
0,1,2022,23,53,36,False,30,13,17,0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,2022,53,79,63,False,26,10,16,1,30.0,NaN,13.0,NaN,17.0,NaN,1
2,2,2022,21,52,35,False,31,14,17,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2,2022,52,84,67,False,32,15,17,1,31.0,NaN,14.0,NaN,17.0,NaN,1
4,3,2022,11,44,30,False,33,19,14,0,NaN,NaN,NaN,NaN,NaN,NaN,0


# Wearable features

In [13]:
computed_temperature = pd.read_csv("computed_temperature.csv")

In [14]:
computed_temperature.head(1)

,id,study_interval,is_weekend,sleep_start_day_in_study,sleep_start_timestamp,sleep_end_day_in_study,sleep_end_timestamp,type,temperature_samples,nightly_temperature,baseline_relative_sample_sum,baseline_relative_sample_sum_of_squares,baseline_relative_nightly_standard_deviation,baseline_relative_sample_standard_deviation
0,1,2022,True,1,00:08:00,1,10:25:30,SKIN,414,34.616087,NaN,NaN,NaN,NaN


In [15]:
computed_temperature = computed_temperature.rename(columns={"sleep_start_day_in_study": "day_in_study"})

In [16]:
computed_temperature = (computed_temperature.sort_values("temperature_samples", ascending=False)
            .drop_duplicates(KEY, keep="first"))

In [17]:
resting_heart_rate=pd.read_csv("resting_heart_rate.csv")

In [18]:
resting_heart_rate.head(1)

,id,study_interval,is_weekend,day_in_study,value,error
0,1,2022,True,1,74.785346,100.0


In [19]:
heart_rate_variability_details=pd.read_csv("heart_rate_variability_details.csv")

In [20]:
KEY = ["id", "study_interval", "day_in_study"]
merged_1= computed_temperature.merge(resting_heart_rate, on=KEY, how="outer")

In [21]:
merged_1.head()

,id,study_interval,is_weekend_x,day_in_study,sleep_start_timestamp,sleep_end_day_in_study,sleep_end_timestamp,type,temperature_samples,nightly_temperature,baseline_relative_sample_sum,baseline_relative_sample_sum_of_squares,baseline_relative_nightly_standard_deviation,baseline_relative_sample_standard_deviation,is_weekend_y,value,error
0,1,2022,True,1,00:08:00,1.0,10:25:30,SKIN,414.0,34.616087,NaN,NaN,NaN,NaN,True,74.785346,100.000000
1,1,2022,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,80.407307,29.833838
2,1,2022,False,3,00:14:00,3.0,09:04:00,SKIN,353.0,34.634929,6.651304,1554.843599,0.487865,2.101622,False,84.686869,24.267298
3,1,2022,False,4,00:12:30,4.0,07:42:00,SKIN,446.0,34.050056,-126.224891,1161.909368,0.424570,1.839029,False,83.852219,10.344703
4,1,2022,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,0.000000,0.000000


In [22]:
heart_rate_variability_details.head(1)

,id,study_interval,is_weekend,day_in_study,timestamp,rmssd,coverage,low_frequency,high_frequency
0,2,2022,True,7,23:25:00,37.058,0.763,817.292,257.772


In [ ]:
# one row per day
KEY = ["id", "study_interval", "day_in_study"]

heart_rate_variability_daily = (heart_rate_variability_details.groupby(KEY, as_index=False)
                .agg(hrv_rmssd=("rmssd", "mean"),
                     hrv_coverage=("coverage", "mean")))

In [24]:
merged_2 = merged_1.merge(heart_rate_variability_daily, on=KEY, how="outer")

In [25]:
merged_2.head(1)

,id,study_interval,is_weekend_x,day_in_study,sleep_start_timestamp,sleep_end_day_in_study,sleep_end_timestamp,type,temperature_samples,nightly_temperature,baseline_relative_sample_sum,baseline_relative_sample_sum_of_squares,baseline_relative_nightly_standard_deviation,baseline_relative_sample_standard_deviation,is_weekend_y,value,error,hrv_rmssd,hrv_coverage
0,1,2022,True,1,00:08:00,1.0,10:25:30,SKIN,414.0,34.616087,NaN,NaN,NaN,NaN,True,74.785346,100.0,NaN,NaN


In [ ]:
# primary ovulation signal — the mean nightly temperature deviation from baseline
merged_2["temp_rel"] = merged_2["baseline_relative_sample_sum"] / merged_2["temperature_samples"]

In [27]:
merged_2 = merged_2.rename(columns={"value": "resting_hr"})

In [ ]:
# average each wearable signal across the cycle's days
def features_for_cycle(row, panel):
    w = panel[(panel.id == row.id) &
              (panel.study_interval == row.study_interval) &
              (panel.day_in_study >= row.onset_day) &
              (panel.day_in_study < row.next_onset_day)]
    return pd.Series({
        "temp_rel_mean": w["temp_rel"].mean(),
        "resting_hr_mean": w["resting_hr"].mean(),
        "hrv_rmssd_mean": w["hrv_rmssd"].mean(),
    })

cycle_features = cycle_seq.apply(lambda r: features_for_cycle(r, merged_2), axis=1)
cycle_seq = pd.concat([cycle_seq, cycle_features], axis=1)

# Other parameters

In [29]:
wrist_temperature=pd.read_csv("wrist_temperature.csv")
wrist_temperature.head(1)

,id,study_interval,is_weekend,day_in_study,timestamp,temperature_diff_from_baseline
0,1,2022,False,3,00:00:00,0.023913


In [30]:
KEY = ["id", "study_interval", "day_in_study"]

wrist_daily = (wrist_temperature.groupby(KEY, as_index=False)
                    .agg(wrist_temp_diff=("temperature_diff_from_baseline", "mean")))

In [31]:
respiratory_rate_summary=pd.read_csv("respiratory_rate_summary.csv")
respiratory_rate_summary.head(1)

,id,study_interval,is_weekend,day_in_study,timestamp,full_sleep_breathing_rate,full_sleep_standard_deviation,full_sleep_signal_to_noise,deep_sleep_breathing_rate,deep_sleep_standard_deviation,deep_sleep_signal_to_noise,light_sleep_breathing_rate,light_sleep_standard_deviation,light_sleep_signal_to_noise,rem_sleep_breathing_rate,rem_sleep_standard_deviation,rem_sleep_signal_to_noise
0,2,2022,True,8,08:20:30,15.6,1.3,17.943,15.6,1.3,17.943,15.0,1.6,14.618,15.6,1.4,5.826


In [32]:
respiratory_rate_summary = respiratory_rate_summary[KEY + ["full_sleep_breathing_rate"]].rename(
        columns={"full_sleep_breathing_rate": "resp_rate"})

In [33]:
sleep=pd.read_csv("sleep.csv")
sleep.head(1)

,id,study_interval,is_weekend,sleep_start_day_in_study,sleep_start_timestamp,sleep_end_day_in_study,sleep_end_timestamp,duration,minutestofallasleep,minutesasleep,minutesawake,minutesafterwakeup,timeinbed,efficiency,type,infocode,levels,mainsleep
0,1,2022,False,4,00:12:30,4,07:42:00,26940000,0,384,65,4,449,93,stages,0,"{'summary': {'deep': {'count': 4, 'minutes': 9...",True


In [34]:
sleep = sleep.rename(columns={"sleep_start_day_in_study": "day_in_study"})

sleep = (sleep[sleep["mainsleep"] == True]
            .sort_values("minutesasleep", ascending=False)
            .drop_duplicates(KEY, keep="first"))

In [35]:
result = (wrist_daily
          .merge(respiratory_rate_summary,  on=KEY, how="outer")
          .merge(sleep, on=KEY, how="outer"))

In [36]:
result.head()

,id,study_interval,day_in_study,wrist_temp_diff,resp_rate,is_weekend,sleep_start_timestamp,sleep_end_day_in_study,sleep_end_timestamp,duration,minutestofallasleep,minutesasleep,minutesawake,minutesafterwakeup,timeinbed,efficiency,type,infocode,levels,mainsleep
0,1,2022,1,NaN,NaN,True,00:08:00,1.0,10:25:30,37020000.0,0.0,596.0,21.0,0.0,617.0,97.0,classic,1.0,"{'summary': {'restless': {'count': 8, 'minutes...",True
1,1,2022,3,-2.410090,NaN,False,00:14:00,3.0,09:04:00,31800000.0,0.0,502.0,28.0,0.0,530.0,95.0,classic,1.0,"{'summary': {'restless': {'count': 14, 'minute...",True
2,1,2022,4,-2.198042,NaN,False,00:12:30,4.0,07:42:00,26940000.0,0.0,384.0,65.0,4.0,449.0,93.0,stages,0.0,"{'summary': {'deep': {'count': 4, 'minutes': 9...",True
3,1,2022,6,-1.609594,NaN,False,00:22:00,6.0,06:38:00,22560000.0,0.0,365.0,11.0,0.0,376.0,97.0,classic,1.0,"{'summary': {'restless': {'count': 6, 'minutes...",True
4,1,2022,7,-0.877416,NaN,True,01:26:00,7.0,10:04:30,31080000.0,0.0,467.0,51.0,0.0,518.0,97.0,stages,0.0,"{'summary': {'deep': {'count': 7, 'minutes': 8...",True


In [37]:
merged_3 = (merged_2.merge(result, on=KEY, how="outer"))
            

In [38]:
merged_3.head()

,id,study_interval,is_weekend_x,day_in_study,sleep_start_timestamp_x,sleep_end_day_in_study_x,sleep_end_timestamp_x,type_x,temperature_samples,nightly_temperature,...,minutestofallasleep,minutesasleep,minutesawake,minutesafterwakeup,timeinbed,efficiency,type_y,infocode,levels,mainsleep
0,1,2022,True,1,00:08:00,1.0,10:25:30,SKIN,414.0,34.616087,...,0.0,596.0,21.0,0.0,617.0,97.0,classic,1.0,"{'summary': {'restless': {'count': 8, 'minutes...",True
1,1,2022,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2022,False,3,00:14:00,3.0,09:04:00,SKIN,353.0,34.634929,...,0.0,502.0,28.0,0.0,530.0,95.0,classic,1.0,"{'summary': {'restless': {'count': 14, 'minute...",True
3,1,2022,False,4,00:12:30,4.0,07:42:00,SKIN,446.0,34.050056,...,0.0,384.0,65.0,4.0,449.0,93.0,stages,0.0,"{'summary': {'deep': {'count': 4, 'minutes': 9...",True
4,1,2022,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# statistic covariates

In [39]:
subject_info=pd.read_csv("subject_info.csv")
subject_info.head(1)

,id,birth_year,gender,ethnicity,education,sexually_active,self_report_menstrual_health_literacy,age_of_first_menarche
0,1,1999,Woman,White,"Some university/ post-secondary, no degree",Yes,NaN,14


In [40]:
subj = subject_info[["id", "birth_year", "age_of_first_menarche", "gender"]].copy()

In [41]:
height_and_weight=pd.read_csv("height_and_weight.csv")
height_and_weight.head(1)

,id,height_2022,weight_2022,height_2024,weight_2024
0,1,NaN,NaN,NaN,NaN


In [42]:
hw = pd.wide_to_long(height_and_weight, stubnames=["height", "weight"],
                     i="id", j="study_interval", sep="_").reset_index()
# now: id | study_interval | height | weight
hw["bmi"] = hw["weight"] / (hw["height"] / 100) ** 2
hw = hw[["id", "study_interval", "bmi"]]

In [43]:
merged_3 = merged_3.merge(subj, on="id", how="left")
merged_3 = merged_3.merge(hw, on=["id", "study_interval"], how="left")
merged_3["age"] = merged_3["study_interval"] - merged_3["birth_year"]

# Cofounders

In [44]:
steps=pd.read_csv("steps.csv")
steps.head(1)

,id,study_interval,is_weekend,day_in_study,timestamp,steps
0,1,2022,True,1,04:42:00,0


In [45]:
KEY = ["id", "study_interval", "day_in_study"]

steps_daily = (steps.groupby(KEY, as_index=False)
                    .agg(steps_total=("steps", "sum")))

In [46]:
active_minutes=pd.read_csv("active_minutes.csv")
active_minutes.head(1)

,id,study_interval,is_weekend,day_in_study,sedentary,lightly,moderately,very
0,1,2022,True,1,753.0,64,0,0


In [47]:
active = active_minutes[KEY + ["sedentary", "lightly", "moderately", "very"]].copy()
active["active_minutes"] = active["lightly"] + active["moderately"] + active["very"]
active = active[KEY + ["active_minutes"]]   

In [48]:
calories=pd.read_csv("calories.csv")
calories.head(1)

,id,study_interval,is_weekend,day_in_study,timestamp,calories
0,1,2022,True,1,00:00:00,1.0


In [49]:
cal_daily = (calories.groupby(KEY, as_index=False)
                     .agg(calories_total=("calories", "sum")))

In [50]:
merged_3 = (merged_3
            .merge(steps_daily, on=KEY, how="left")
            .merge(active,      on=KEY, how="left")
            .merge(cal_daily,   on=KEY, how="left"))

In [51]:
merged_3.head()

,id,study_interval,is_weekend_x,day_in_study,sleep_start_timestamp_x,sleep_end_day_in_study_x,sleep_end_timestamp_x,type_x,temperature_samples,nightly_temperature,...,levels,mainsleep,birth_year,age_of_first_menarche,gender,bmi,age,steps_total,active_minutes,calories_total
0,1,2022,True,1,00:08:00,1.0,10:25:30,SKIN,414.0,34.616087,...,"{'summary': {'restless': {'count': 8, 'minutes...",True,1999,14,Woman,NaN,23,992.0,64.0,1542.0
1,1,2022,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1999,14,Woman,NaN,23,838.0,74.0,1591.0
2,1,2022,False,3,00:14:00,3.0,09:04:00,SKIN,353.0,34.634929,...,"{'summary': {'restless': {'count': 14, 'minute...",True,1999,14,Woman,NaN,23,2586.0,159.0,1755.0
3,1,2022,False,4,00:12:30,4.0,07:42:00,SKIN,446.0,34.050056,...,"{'summary': {'deep': {'count': 4, 'minutes': 9...",True,1999,14,Woman,NaN,23,1275.0,86.0,1552.0
4,1,2022,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1999,14,Woman,NaN,23,436.0,10.0,1456.0


In [145]:
len(set(merged_3["id"].to_list()))

42

In [52]:
keep = KEY + [
    "temp_rel", "nightly_temperature", "wrist_temp_diff",
    "resting_hr", "hrv_rmssd", "resp_rate",
    "minutesasleep", "efficiency",
    "steps_total", "active_minutes", "calories_total",
    "age", "bmi", "age_of_first_menarche", "gender",
]
panel = merged_3[[c for c in keep if c in merged_3.columns]].copy()

In [133]:
panel.head()

,id,study_interval,day_in_study,temp_rel,nightly_temperature,wrist_temp_diff,resting_hr,hrv_rmssd,resp_rate,minutesasleep,efficiency,steps_total,active_minutes,calories_total,age,bmi,age_of_first_menarche,gender,nt_rel,temp_signal
0,1,2022,1,NaN,34.616087,NaN,74.785346,NaN,NaN,596.0,97.0,992.0,64.0,1542.0,23,NaN,14,Woman,0.085428,0.085428
1,1,2022,2,NaN,NaN,NaN,80.407307,NaN,NaN,NaN,NaN,838.0,74.0,1591.0,23,NaN,14,Woman,NaN,NaN
2,1,2022,3,0.018842,34.634929,-2.410090,84.686869,NaN,NaN,502.0,95.0,2586.0,159.0,1755.0,23,NaN,14,Woman,0.104270,0.018842
3,1,2022,4,-0.283015,34.050056,-2.198042,83.852219,NaN,NaN,384.0,93.0,1275.0,86.0,1552.0,23,NaN,14,Woman,-0.480603,-0.283015
4,1,2022,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,436.0,10.0,1456.0,23,NaN,14,Woman,NaN,NaN


In [53]:
panel.set_index(KEY).notna().mean().sort_values()

bmi                      0.373788
temp_rel                 0.570239
wrist_temp_diff          0.699625
active_minutes           0.849659
nightly_temperature      0.864778
resp_rate                0.960000
hrv_rmssd                0.971502
minutesasleep            0.976621
efficiency               0.976621
steps_total              0.995222
calories_total           0.999352
resting_hr               0.999488
age                      1.000000
age_of_first_menarche    1.000000
gender                   1.000000
dtype: float64

In [54]:
len(panel)

29300

In [55]:
panel.head()

,id,study_interval,day_in_study,temp_rel,nightly_temperature,wrist_temp_diff,resting_hr,hrv_rmssd,resp_rate,minutesasleep,efficiency,steps_total,active_minutes,calories_total,age,bmi,age_of_first_menarche,gender
0,1,2022,1,NaN,34.616087,NaN,74.785346,NaN,NaN,596.0,97.0,992.0,64.0,1542.0,23,NaN,14,Woman
1,1,2022,2,NaN,NaN,NaN,80.407307,NaN,NaN,NaN,NaN,838.0,74.0,1591.0,23,NaN,14,Woman
2,1,2022,3,0.018842,34.634929,-2.410090,84.686869,NaN,NaN,502.0,95.0,2586.0,159.0,1755.0,23,NaN,14,Woman
3,1,2022,4,-0.283015,34.050056,-2.198042,83.852219,NaN,NaN,384.0,93.0,1275.0,86.0,1552.0,23,NaN,14,Woman
4,1,2022,5,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,436.0,10.0,1456.0,23,NaN,14,Woman


In [74]:
cycle_seq.head()

,id,study_interval,onset_day,next_onset_day,ovulation_day,pdg_confirmed,cycle_length,follicular_length,luteal_length,cycle_index,prior_cycle_length_mean,prior_cycle_length_sd,prior_follicular_length_mean,prior_follicular_length_sd,prior_luteal_length_mean,prior_luteal_length_sd,n_prior_cycles,temp_rel_mean,resting_hr_mean,hrv_rmssd_mean
0,1,2022,23,53,36,False,30,13,17,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-0.465553,39.374085,NaN
1,1,2022,53,79,63,False,26,10,16,1,30.0,NaN,13.0,NaN,17.0,NaN,1,-0.350834,17.424904,NaN
2,2,2022,21,52,35,False,31,14,17,0,NaN,NaN,NaN,NaN,NaN,NaN,0,-0.222367,61.559370,44.679115
3,2,2022,52,84,67,False,32,15,17,1,31.0,NaN,14.0,NaN,17.0,NaN,1,0.117419,20.794889,44.625501
4,3,2022,16,44,30,False,28,14,14,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0.061877,64.296216,NaN


In [ ]:
panel["resting_hr"].describe()   

count    29285.000000
mean        61.700094
std         25.333109
min          0.000000
25%         61.976951
50%         70.569468
75%         74.768438
max         89.345984
Name: resting_hr, dtype: float64

In [58]:
panel.loc[panel["resting_hr"] < 30, "resting_hr"] = pd.NA

In [60]:
# re-run your features_for_cycle windowing on the cleaned panel, then:
panel["resting_hr"].describe()

count    25364.000000
mean        71.238261
std          7.840358
min         47.562128
25%         66.367813
50%         71.879683
75%         75.649695
max         89.345984
Name: resting_hr, dtype: float64

In [62]:
panel.columns

Index(['id', 'study_interval', 'day_in_study', 'temp_rel',
       'nightly_temperature', 'wrist_temp_diff', 'resting_hr', 'hrv_rmssd',
       'resp_rate', 'minutesasleep', 'efficiency', 'steps_total',
       'active_minutes', 'calories_total', 'age', 'bmi',
       'age_of_first_menarche', 'gender'],
      dtype='str')

In [81]:
cycle_seq.columns

Index(['id', 'study_interval', 'onset_day', 'next_onset_day', 'ovulation_day',
       'pdg_confirmed', 'cycle_length', 'follicular_length', 'luteal_length',
       'cycle_index', 'prior_cycle_length_mean', 'prior_cycle_length_sd',
       'prior_follicular_length_mean', 'prior_follicular_length_sd',
       'prior_luteal_length_mean', 'prior_luteal_length_sd', 'n_prior_cycles',
       'temp_rel_mean', 'resting_hr_mean', 'hrv_rmssd_mean'],
      dtype='str')

In [75]:
cycle_seq[["cycle_length", "follicular_length", "luteal_length"]].describe()

,cycle_length,follicular_length,luteal_length
count,61.000000,61.000000,61.000000
mean,29.393443,15.049180,14.344262
std,4.879476,4.652692,2.857069
min,20.000000,5.000000,7.000000
25%,26.000000,12.000000,13.000000
50%,29.000000,14.000000,15.000000
75%,31.000000,17.000000,16.000000
max,43.000000,28.000000,19.000000


In [80]:
cycle_seq["ovulation_day"] = cycle_seq["onset_day"] + cycle_seq["follicular_length"]

In [130]:
cycle_seq["fertile_start"] = cycle_seq["ovulation_day"] - 5
cycle_seq["fertile_end"]   = cycle_seq["ovulation_day"]

In [136]:
cycle_seq.head(1)

,id,study_interval,onset_day,next_onset_day,ovulation_day,pdg_confirmed,cycle_length,follicular_length,luteal_length,cycle_index,...,prior_follicular_length_mean,prior_follicular_length_sd,prior_luteal_length_mean,prior_luteal_length_sd,n_prior_cycles,temp_rel_mean,resting_hr_mean,hrv_rmssd_mean,fertile_start,fertile_end
0,1,2022,23,53,36,False,30,13,17,0,...,NaN,NaN,NaN,NaN,0,-0.465553,39.374085,NaN,31,36


In [134]:
cycle_seq.to_csv("cycle_seq.csv")

In [1]:
import pandas as pd
cycle_seq=pd.read_csv("cycle_seq.csv",index_col=False)
panel=pd.read_csv("panel.csv",index_col=False)

In [2]:
panel.head(1)

,id,study_interval,day_in_study,temp_rel,nightly_temperature,wrist_temp_diff,resting_hr,hrv_rmssd,resp_rate,minutesasleep,efficiency,steps_total,active_minutes,calories_total,age,bmi,age_of_first_menarche,gender
0,1,2022,1,NaN,34.616087,NaN,74.785346,NaN,NaN,596.0,97.0,992.0,64.0,1542.0,23,NaN,14,Woman


In [135]:
panel.to_csv("panel.csv")

In [3]:
KEYS = ["id", "study_interval", "cycle_index"]

# ---- LABELS: survival outcomes (from cycle_seq) ----
LABEL_COLS = [
    "onset_day", "next_onset_day", "ovulation_day",
    "cycle_length", "follicular_length", "luteal_length",
    "pdg_confirmed",
]
labels = cycle_seq[["id", "cycle_index"] + LABEL_COLS].copy()

# ---- CYCLE-LEVEL FEATURES (safe history, from cycle_seq) ----
PRIOR_FEATS = [
    "prior_cycle_length_mean", "prior_cycle_length_sd",
    "prior_follicular_length_mean", "prior_follicular_length_sd",
    "prior_luteal_length_mean", "prior_luteal_length_sd",
    "n_prior_cycles",
]
cycle_features = cycle_seq[["id", "cycle_index"] + PRIOR_FEATS].copy()

# ---- LEAKY (whole-cycle means) — quarantined, not in features ----
LEAKY = ["temp_rel_mean", "resting_hr_mean", "hrv_rmssd_mean"]

# ---- DAILY FEATURES (from panel) ----
# static demographics ride along here (constant per id)
features_daily = panel.copy()

# Longitudinal survival rate models

In [4]:
import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupKFold
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv


DAILY = [
    "temp_rel",
    "nightly_temperature",
    "wrist_temp_diff",
    "resting_hr",
    "hrv_rmssd",
    "resp_rate",
    "minutesasleep",
    "efficiency",
    "steps_total",
    "active_minutes",
    "calories_total",
]

HISTORY = [
    "prior_cycle_length_mean",
    "prior_cycle_length_sd",
    "n_prior_cycles",
]


class CyclePredictor:
    def __init__(self, event="menses"):
        self.event = event

        self.end = (
            "next_onset_day"
            if event == "menses"
            else "ovulation_day"
        )

        self.rsf = RandomSurvivalForest(
            n_estimators=300,
            min_samples_leaf=8,
            max_depth=5,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1,
        )

    # ---------------------------------------------------------
    # Features
    # ---------------------------------------------------------

    def _feat(self, cd, hist, t):
        sofar = (
            cd[cd["day"] <= t]
            .copy()
            .sort_values("day")
        )

        row = {
            "day_now": float(t),

            # Safe cycle-level information known at cycle start
            "cycle_index": pd.to_numeric(
                hist.get("cycle_index", np.nan),
                errors="coerce",
            ),
        }

        for c in DAILY:
            if c in sofar.columns:
                observed = sofar[
                    ["day", c]
                ].copy()

                observed["day"] = pd.to_numeric(
                    observed["day"],
                    errors="coerce",
                )

                observed[c] = pd.to_numeric(
                    observed[c],
                    errors="coerce",
                )

                observed = (
                    observed.dropna()
                    .sort_values("day")
                )
            else:
                observed = pd.DataFrame(
                    columns=["day", c]
                )

            n = len(observed)

            # Missing-data information
            row[c + "_missing"] = float(n == 0)

            # Percentage of possible days with measurements
            row[c + "_coverage"] = float(
                n / max(t + 1, 1)
            )

            if n == 0:
                row[c + "_mean"] = np.nan
                row[c + "_last"] = np.nan
                row[c + "_slope"] = np.nan
                continue

            measurement_days = observed[
                "day"
            ].to_numpy(dtype=float)

            values = observed[
                c
            ].to_numpy(dtype=float)

            row[c + "_mean"] = float(
                np.mean(values)
            )

            row[c + "_last"] = float(
                values[-1]
            )

            # Use actual observation days for the slope
            if (
                n >= 2
                and np.unique(measurement_days).size >= 2
            ):
                row[c + "_slope"] = float(
                    np.polyfit(
                        measurement_days,
                        values,
                        1,
                    )[0]
                )
            else:
                row[c + "_slope"] = np.nan

        for h in HISTORY:
            value = pd.to_numeric(
                hist.get(h, np.nan),
                errors="coerce",
            )

            row[h] = value
            row[h + "_missing"] = float(
                pd.isna(value)
            )

        return row

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------

    def fit(
        self,
        cs,
        pn,
        cf,
        censor_day_col=None,
    ):
        seq = cs.copy()
        panel = pn.copy()

        seq["onset_day"] = pd.to_numeric(
            seq["onset_day"],
            errors="coerce",
        )

        seq[self.end] = pd.to_numeric(
            seq[self.end],
            errors="coerce",
        )

        panel["day_in_study"] = pd.to_numeric(
            panel["day_in_study"],
            errors="coerce",
        )

        # Find the following menstrual onset
        seq = (
            seq.sort_values(
                ["id", "onset_day"]
            )
            .copy()
        )

        seq["_following_onset"] = (
            seq.groupby("id")["onset_day"]
            .shift(-1)
        )

        if (
            censor_day_col is not None
            and censor_day_col in seq.columns
        ):
            seq[censor_day_col] = pd.to_numeric(
                seq[censor_day_col],
                errors="coerce",
            )

        H = (
            cf.set_index(
                ["id", "cycle_index"]
            )
            .to_dict("index")
        )

        rows = []
        times = []
        events = []
        groups = []

        for _, cycle in seq.iterrows():
            wid = cycle["id"]

            onset = cycle["onset_day"]
            event_day = cycle[self.end]
            following_onset = cycle[
                "_following_onset"
            ]

            if pd.isna(onset):
                continue

            event_observed = False
            stop_day = np.nan

            # Known event
            if (
                pd.notna(event_day)
                and event_day > onset
            ):
                stop_day = event_day
                event_observed = True

            # Missing next_onset_day, but next cycle onset exists
            elif (
                self.event == "menses"
                and pd.notna(following_onset)
                and following_onset > onset
            ):
                stop_day = following_onset
                event_observed = True

            # Missing event: retain as censored
            else:
                censor_candidates = []

                if (
                    censor_day_col is not None
                    and censor_day_col in cycle.index
                ):
                    explicit_censor = cycle[
                        censor_day_col
                    ]

                    if (
                        pd.notna(explicit_censor)
                        and explicit_censor > onset
                    ):
                        censor_candidates.append(
                            float(explicit_censor)
                        )

                available_days = panel.loc[
                    (panel["id"] == wid)
                    & (
                        panel["day_in_study"]
                        >= onset
                    ),
                    "day_in_study",
                ].dropna()

                # Prevent measurements from the next cycle
                if pd.notna(following_onset):
                    available_days = available_days[
                        available_days
                        < following_onset
                    ]

                    if following_onset > onset:
                        censor_candidates.append(
                            float(following_onset)
                        )

                if not available_days.empty:
                    censor_candidates.append(
                        float(available_days.max())
                    )

                if censor_candidates:
                    stop_day = min(
                        censor_candidates
                    )

            # Cannot use a cycle without event/censoring time
            if (
                pd.isna(stop_day)
                or stop_day <= onset
            ):
                continue

            total = int(
                np.floor(stop_day - onset)
            )

            if total <= 2:
                continue

            if event_observed:
                days = panel[
                    (panel["id"] == wid)
                    & (
                        panel["day_in_study"]
                        >= onset
                    )
                    & (
                        panel["day_in_study"]
                        < stop_day
                    )
                ].copy()
            else:
                days = panel[
                    (panel["id"] == wid)
                    & (
                        panel["day_in_study"]
                        >= onset
                    )
                    & (
                        panel["day_in_study"]
                        <= stop_day
                    )
                ].copy()

            # Keep the cycle even when all sensor data is missing
            if days.empty:
                days = pd.DataFrame(
                    columns=list(panel.columns)
                    + ["day"]
                )
            else:
                days["day"] = (
                    days["day_in_study"]
                    - onset
                )

            cycle_index = cycle.get(
                "cycle_index",
                np.nan,
            )

            history = dict(
                H.get(
                    (wid, cycle_index),
                    {},
                )
            )

            # Add safe cycle-level feature
            history["cycle_index"] = cycle_index

            for t in range(2, total, 2):
                remaining = float(
                    stop_day - (onset + t)
                )

                if remaining <= 0:
                    continue

                rows.append(
                    self._feat(
                        days,
                        history,
                        t,
                    )
                )

                times.append(remaining)
                events.append(event_observed)
                groups.append(wid)

        if not rows:
            raise ValueError(
                "No training observations were created."
            )

        X_raw = pd.DataFrame(rows).replace(
            [np.inf, -np.inf],
            np.nan,
        )

        self.features_ = X_raw.columns.tolist()

        self.fill_ = (
            X_raw.median()
            .reindex(self.features_)
            .fillna(0.0)
        )

        X = X_raw.fillna(self.fill_)

        y = Surv.from_arrays(
            event=np.asarray(
                events,
                dtype=bool,
            ),
            time=np.asarray(
                times,
                dtype=float,
            ),
        )

        self._X_raw = X_raw
        self._X = X
        self._y = y
        self._g = np.asarray(groups)

        self.rsf.fit(X, y)

        self.train_c_ = self.rsf.score(
            X,
            y,
        )

        return self

    # ---------------------------------------------------------
    # Cross-validation
    # ---------------------------------------------------------

    def cv(self, n=5):
        unique_people = pd.Series(
            self._g
        ).nunique()

        n = min(n, unique_people)

        splitter = GroupKFold(
            n_splits=n
        )

        scores = []

        for train, test in splitter.split(
            self._X_raw,
            self._y,
            self._g,
        ):
            X_train_raw = self._X_raw.iloc[
                train
            ]

            X_test_raw = self._X_raw.iloc[
                test
            ]

            # Calculate medians only from training fold
            fold_fill = (
                X_train_raw.median()
                .reindex(self.features_)
                .fillna(0.0)
            )

            X_train = X_train_raw.fillna(
                fold_fill
            )

            X_test = X_test_raw.fillna(
                fold_fill
            )

            model = RandomSurvivalForest(
                n_estimators=300,
                min_samples_leaf=8,
                max_depth=5,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            )

            model.fit(
                X_train,
                self._y[train],
            )

            try:
                score = model.score(
                    X_test,
                    self._y[test],
                )
            except ValueError:
                score = np.nan

            scores.append(score)

        return np.asarray(scores)

    # ---------------------------------------------------------
    # Feature importance
    # ---------------------------------------------------------

    def importances(
        self,
        top=12,
        n_repeats=10,
    ):
        result = permutation_importance(
            self.rsf,
            self._X,
            self._y,
            n_repeats=n_repeats,
            random_state=42,
            n_jobs=-1,
        )

        output = pd.DataFrame(
            {
                "importance_mean":
                    result.importances_mean,
                "importance_sd":
                    result.importances_std,
            },
            index=self.features_,
        )

        output.index.name = "feature"

        return (
            output
            .sort_values(
                "importance_mean",
                ascending=False,
            )
            .head(top)
        )

    # ---------------------------------------------------------
    # Prediction
    # ---------------------------------------------------------

    def predict_day(
        self,
        cd,
        hist,
        t,
    ):
        features = self._feat(
            cd,
            hist,
            t,
        )

        X = (
            pd.DataFrame([features])
            .reindex(columns=self.features_)
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .fillna(self.fill_)
        )

        fn = self.rsf.predict_survival_function(
            X
        )[0]

        remaining = np.asarray(
            fn.x,
            dtype=float,
        )

        survival = np.asarray(
            fn.y,
            dtype=float,
        )

        probability = -np.diff(
            np.r_[1.0, survival]
        )

        return pd.DataFrame(
            {
                "day": t + remaining,
                "p_event_day": np.round(
                    probability,
                    4,
                ),
                "surv": np.round(
                    survival,
                    4,
                ),
            }
        )

In [5]:
model = CyclePredictor(
    event="menses"
)

model.fit(
    cycle_seq,
    panel,
    cycle_features,
)

model_ov = CyclePredictor(
    event="ovulation"
)

model_ov.fit(
    cycle_seq,
    panel,
    cycle_features,
)

In [ ]:
print("Menses training C-index:", model.train_c_)
print("Ovulation training C-index:", model_ov.train_c_)

print(model_ov.features_)

In [ ]:
model_ov = CyclePredictor(event="ovulation")

model_ov.fit(
    cycle_seq,
    panel,
    cycle_features,
    infer_censor_from_panel=True,
)

In [ ]:
def forecast_fertility(
    model,
    cycle_seq,
    panel,
    cycle_features,
    wid,
    cycle_idx=0,
    fertile_before=5,
    fertile_after=1,
):
    # Get this woman's cycles in chronological order
    woman_cycles = (
        cycle_seq[cycle_seq["id"] == wid]
        .copy()
        .sort_values("onset_day")
        .reset_index(drop=True)
    )

    if woman_cycles.empty:
        raise ValueError(f"No cycles found for woman {wid}")

    if cycle_idx >= len(woman_cycles):
        raise IndexError(
            f"cycle_idx={cycle_idx}, but woman {wid} "
            f"has only {len(woman_cycles)} cycles"
        )

    row = woman_cycles.iloc[cycle_idx]

    onset = pd.to_numeric(row["onset_day"], errors="coerce")
    ovday = pd.to_numeric(row.get("ovulation_day"), errors="coerce")
    end = pd.to_numeric(row.get("next_onset_day"), errors="coerce")
    selected_cycle_index = row.get("cycle_index", cycle_idx)

    if pd.isna(onset):
        raise ValueError("The selected cycle has no valid onset_day")

    # If next_onset_day is missing, use the following cycle onset
    if pd.isna(end) and cycle_idx + 1 < len(woman_cycles):
        end = pd.to_numeric(
            woman_cycles.iloc[cycle_idx + 1]["onset_day"],
            errors="coerce",
        )

    # Get this woman's panel data after cycle onset
    days = panel[
        (panel["id"] == wid)
        & (panel["day_in_study"] >= onset)
    ].copy()

    # Stop at next onset when it is known
    if pd.notna(end) and end > onset:
        days = days[
            days["day_in_study"] < end
        ].copy()

    if days.empty:
        raise ValueError(
            f"No panel data found for woman {wid}, cycle {cycle_idx}"
        )

    days["day"] = (
        pd.to_numeric(days["day_in_study"], errors="coerce")
        - onset
    )

    days = (
        days.dropna(subset=["day"])
        .sort_values("day")
    )

    # ---------------------------------------------------------
    # Use features belonging to the correct cycle
    # ---------------------------------------------------------

    hist_rows = cycle_features[
        (cycle_features["id"] == wid)
        & (
            cycle_features["cycle_index"]
            == selected_cycle_index
        )
    ]

    if hist_rows.empty:
        hist = {}
    else:
        hist = hist_rows.iloc[0].to_dict()

    # Also use matching cycle-level information when a HISTORY
    # feature exists in cycle_seq but is missing in cycle_features
    for feature in HISTORY:
        current_value = hist.get(feature, np.nan)

        if pd.isna(current_value) and feature in row.index:
            hist[feature] = row[feature]

    # Make sure every expected history feature exists
    for feature in HISTORY:
        hist.setdefault(feature, np.nan)

    # True ovulation is used only for evaluation/printing
    if pd.notna(ovday) and ovday > onset:
        true_ov = int(round(ovday - onset))
    else:
        true_ov = None

    maximum_available_day = int(
        np.floor(days["day"].max())
    )

    if true_ov is not None:
        final_forecast_day = min(
            maximum_available_day,
            true_ov + 2,
        )
    else:
        final_forecast_day = maximum_available_day

    results = []

    for today in range(3, final_forecast_day + 1):

        # Important: give the model only data available today
        days_so_far = days[
            days["day"] <= today
        ].copy()

        prediction = model.predict_day(
            cycle_days=days_so_far,
            history=hist,
            t=today,
        )

        prediction = prediction[
            prediction["day"] >= today
        ]

        if prediction.empty:
            continue

        peak_row = prediction.loc[
            prediction["p_event_day"].idxmax()
        ]

        predicted_ovulation = float(
            peak_row["day"]
        )

        days_away = predicted_ovulation - today

        fertile_now = (
            predicted_ovulation - fertile_before
            <= today
            <= predicted_ovulation + fertile_after
        )

        results.append(
            {
                "woman_id": wid,
                "cycle_index": selected_cycle_index,
                "today": today,
                "predicted_ovulation_day": round(
                    predicted_ovulation,
                    1,
                ),
                "days_until_predicted_ovulation": round(
                    days_away,
                    1,
                ),
                "actual_ovulation_day": true_ov,
                "fertile_window_now": fertile_now,
            }
        )

    result = pd.DataFrame(results)

    print(
        f"Woman {wid}, cycle {selected_cycle_index}"
    )

    if true_ov is not None:
        print(f"Actual ovulation: cycle day {true_ov}")
    else:
        print("Actual ovulation: unavailable")

    display(result)

    return result

In [ ]:
fertility_result = forecast_fertility(
    model=model_ov,
    cycle_seq=cycle_seq,
    panel=panel,
    cycle_features=cycle_features,
    wid=1,
    cycle_idx=0,
)